# RISC-V + partially-reconfigurable CFU — PYNQ-Z2 demo

Run top to bottom. Expected readback of dmem words 0,4,5,8:

| slot contents | readback |
|---|---|
| multiplier (`config1`) | `[13, 42, 0, 0]` |
| bitwise (`config2`) | `[13, 2, 6, 1]` |

`ctl[9:0]` = dbg_addr, `ctl[10]` = core reset (GPIO ch1); `dbg_data` = dmem[dbg_addr] (GPIO ch2).

Three timing tiers are reported for every load:

1. **PYNQ API** — `Bitstream(...).download()`: bit→bin conversion + copy to `/lib/firmware` + sysfs + kernel. What a PYNQ user gets.
2. **raw fpga_manager** — sysfs write of the already-converted `.bin`: kernel `zynq-fpga` driver + PCAP DMA only.
3. **PCAP theoretical** — bytes / 400 MB/s (32 bit × 100 MHz). A lower bound, not a prediction.

(1)−(2) is Python + file conversion; (2)−(3) is kernel/driver/DMA overhead.

In [ ]:
import os, statistics, time
from pynq import Overlay, Bitstream
import pynq; print('PYNQ', pynq.__version__)

FULL = 'config1.bit'
PART = {'mul': 'config1_pblock_cfu_partial.bit', 'bitwise': 'config2_pblock_cfu_partial.bit'}
GOLD = {'mul': [13, 42, 0, 0], 'bitwise': [13, 2, 6, 1]}
RST = 1 << 10
PCAP_BPS = 400e6
for f in [FULL, 'config1.hwh', *PART.values()]:
    print(f'{os.path.getsize(f):>9} B  {f}   (PCAP theoretical {os.path.getsize(f)/PCAP_BPS*1e3:.2f} ms)')

## 1. Load the full bitstream (multiplier in the slot)
`Overlay()` also parses the `.hwh` and sets FCLK0; the second line is the fair full-bitstream number (no hwh parse).

In [ ]:
t0 = time.perf_counter(); ol = Overlay(FULL); t_overlay = time.perf_counter() - t0
g = ol.axi_gpio_0
ctl, dbg = g.channel1, g.channel2
print(f'Overlay(FULL)            : {t_overlay*1e3:.1f} ms   (hwh parse + bit->bin + load)')
EXPECT_MHZ = 50.0
print('FCLK0 =', pynq.ps.Clocks.fclk0_mhz, 'MHz')
assert abs(pynq.ps.Clocks.fclk0_mhz - EXPECT_MHZ) < 0.1, 'PL clock is not what the design was timed at -- hwh/PLL mismatch, see ps_bd.tcl'

t0 = time.perf_counter(); Bitstream(FULL).download(); t_full = time.perf_counter() - t0
print(f'Bitstream(FULL).download(): {t_full*1e3:.1f} ms   (tier 1, full)')

## 2. Reset, run, read back

In [ ]:
def run_and_read():
    ctl.write(RST, 0x7FF)          # reset
    ctl.write(0, 0x7FF)            # release; program runs to its self-branch
    time.sleep(0.001)
    out = []
    for a in (0, 4, 5, 8):
        ctl.write(a, 0x3FF)
        out.append(dbg.read())
    return out

r = run_and_read(); print(r, '<- expect', GOLD['mul'])

## 3. One partial swap → bitwise unit
Core is held in reset during the swap, so the slot is idle (no DFX decoupler needed). First swap in each direction pays a one-time bit→bin conversion.

In [ ]:
def fpga_state():
    return open('/sys/class/fpga_manager/fpga0/state').read().strip()

def swap(variant):
    ctl.write(RST, 0x7FF)
    t = time.perf_counter()
    Bitstream(PART[variant], partial=True).download()
    return time.perf_counter() - t

dt = swap('bitwise')
r = run_and_read(); print(r, '<- expect', GOLD['bitwise'], f'  swap took {dt*1e3:.2f} ms')

## 4. Swap back → multiplier

In [ ]:
dt = swap('mul')
r = run_and_read(); print(r, '<- expect', GOLD['mul'], f'  swap took {dt*1e3:.2f} ms')

## 5. Tier 1 — PYNQ API, N swaps, readback checked every time

In [ ]:
N = 20
times = {'mul': [], 'bitwise': []}; misses = []
cur = 'mul'
for i in range(N):
    cur = 'bitwise' if cur == 'mul' else 'mul'
    times[cur].append(swap(cur))
    r = run_and_read()
    if r != GOLD[cur]:                      # intermittent miss seen once (2026-09-18): gather evidence, then retry
        again = run_and_read()
        print(f'MISS swap {i} -> {cur}: got {r}; re-read {again}; fpga state {fpga_state()}')
        misses.append((i, cur, r, again))
        swap(cur); assert run_and_read() == GOLD[cur], 'still wrong after re-download'
print(f'{N} swaps, {len(misses)} misses (each recovered by a re-download)' if misses else f'{N} swaps OK, every readback matched its variant')
for v in ('mul', 'bitwise'):
    ms = [t*1e3 for t in times[v]]
    print(f'tier 1 partial -> {v:8s}: median {statistics.median(ms):.2f} ms  min {min(ms):.2f}  max {max(ms):.2f}')

## 6. Tier 2 — raw fpga_manager write of the `.bin` PYNQ already converted
Partial: `flags=1`. Full: `flags=0` (then the multiplier is back in the slot).

In [ ]:
def fpga_mgr(bitfile, partial):
    binf = os.path.basename(bitfile).replace('.bit', '.bin')
    assert os.path.exists('/lib/firmware/' + binf), binf
    ctl.write(RST, 0x7FF)
    with open('/sys/class/fpga_manager/fpga0/flags', 'w') as f: f.write('1' if partial else '0')
    t = time.perf_counter()
    with open('/sys/class/fpga_manager/fpga0/firmware', 'w') as f: f.write(binf)
    return time.perf_counter() - t

raw = {'mul': [], 'bitwise': []}; raw_misses = []
cur = 'mul'
for i in range(N):
    cur = 'bitwise' if cur == 'mul' else 'mul'
    raw[cur].append(fpga_mgr(PART[cur], True))
    r = run_and_read()
    if r != GOLD[cur]:
        again = run_and_read()
        print(f'MISS raw swap {i} -> {cur}: got {r}; re-read {again}; fpga state {fpga_state()}')
        raw_misses.append((i, cur, r, again))
        fpga_mgr(PART[cur], True); assert run_and_read() == GOLD[cur], 'still wrong after re-download'
print(f'{N} raw swaps, {len(raw_misses)} misses' if raw_misses else f'{N} raw swaps OK')
for v in ('mul', 'bitwise'):
    ms = [t*1e3 for t in raw[v]]
    print(f'tier 2 partial -> {v:8s}: median {statistics.median(ms):.2f} ms  min {min(ms):.2f}  max {max(ms):.2f}')

t_full_raw = fpga_mgr(FULL, False)
r = run_and_read(); print(f'tier 2 full: {t_full_raw*1e3:.1f} ms   readback {r}', 'OK' if r == GOLD['mul'] else 'MISMATCH')

## 7. Summary

In [ ]:
sz_f, sz_p = os.path.getsize(FULL), os.path.getsize(PART['mul'])
p1 = statistics.median([t*1e3 for t in times['mul'] + times['bitwise']])
p2 = statistics.median([t*1e3 for t in raw['mul'] + raw['bitwise']])
print(f'sizes        : full {sz_f} B, partial {sz_p} B, ratio {sz_f/sz_p:.1f}x')
print(f'tier 1 PYNQ  : full {t_full*1e3:.1f} ms, partial {p1:.2f} ms, ratio {t_full*1e3/p1:.1f}x')
print(f'tier 2 fpga_mgr: full {t_full_raw*1e3:.1f} ms, partial {p2:.2f} ms, ratio {t_full_raw*1e3/p2:.1f}x')
print(f'tier 3 PCAP  : full {sz_f/PCAP_BPS*1e3:.2f} ms, partial {sz_p/PCAP_BPS*1e3:.2f} ms (theoretical)')
print(f'misses: tier1 {len(misses)}/{N}, tier2 {len(raw_misses)}/{N}')
print(f'FCLK0 {pynq.ps.Clocks.fclk0_mhz} MHz | PYNQ {pynq.__version__}')